# M4 — RAG Backend (CSE488 Term Project) — Kaggle 2x T4, local LLM

Pipeline: query -> extract metadata filters -> restrict candidate pool -> embed + FAISS search -> **local** LLM generates a grounded recommendation citing retrieved specs -> grounding verification -> precision/recall@k evaluation.

**Changed from the Colab/Groq version:** generation now runs a local Hugging Face model
(`Qwen2.5-7B-Instruct`, 4-bit quantized) on your Kaggle GPUs instead of calling the Groq API —
no rate limits, no API key, fully reproducible. Everything else (filter logic, retrieval, the
grounding-verification bug fixes, the eval harness) is unchanged from the earlier fixed version.

**Kaggle setup checklist before running:**
1. Notebook Settings -> Accelerator -> **GPU T4 x2**.
2. Notebook Settings -> Internet -> **On** (needed once, to download the model from the HF Hub).
3. Add `laptop_chunks_embeddings_with_lineage.parquet` and `eval_queries.csv` as a Kaggle **Dataset**
   input (Add Data), or upload them directly — either way they'll land under `/kaggle/input/<your-dataset-slug>/`.
   Adjust `DATA_DIR` in the config cell below to match.


## 1. Setup


In [1]:
import subprocess, sys

# Removed the strict <18 constraint on pyarrow
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "faiss-cpu", "sentence-transformers", "pandas", "pyarrow>=16",
                "transformers>=4.44", "accelerate", "bitsandbytes"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.0 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'faiss-cpu', 'sentence-transformers', 'pandas', 'pyarrow>=16', 'transformers>=4.44', 'accelerate', 'bitsandbytes'], returncode=0)

In [2]:
import os

DATA_DIR = "/kaggle/input/datasets/mrnotalent/laptop-embedding/"
OUT_DIR = "/kaggle/working/"

PARQUET_PATH = os.path.join(DATA_DIR, "laptop_chunks_embeddings_with_lineage.parquet")

# Prefer the corrected, hand-labeled eval set (real compound filter criteria,
# built from actual price/RAM/CPU/GPU/storage constraints) over the weaker
# auto-generated eval_queries.csv (title+CPU echo queries, doesn't exercise
# filter logic — see M3 report for why this was replaced).
_final_path = os.path.join(DATA_DIR, "eval_queries_final.csv")
_fallback_path = os.path.join(DATA_DIR, "eval_queries.csv")

if os.path.exists(_final_path):
    EVAL_CSV_PATH = _final_path
    print("Using corrected hand-labeled eval set: eval_queries_final.csv")
else:
    EVAL_CSV_PATH = _fallback_path
    print("WARNING: eval_queries_final.csv not found — falling back to the weaker "
          "auto-generated eval_queries.csv. Precision/recall numbers from this fallback "
          "should NOT be reported alongside your M3 numbers, which used the corrected set. "
          "Upload eval_queries_final.csv to the input dataset to fix this.")

assert os.path.exists(PARQUET_PATH), f"Not found: {PARQUET_PATH}"
assert os.path.exists(EVAL_CSV_PATH), f"Not found: {EVAL_CSV_PATH}"

Using corrected hand-labeled eval set: eval_queries_final.csv


In [3]:
import re
import json
import numpy as np
import pandas as pd
import faiss

# --- Fix for: ArrowKeyError: A type extension with name
#     datasets.features.features.Array2DExtensionType already defined ---
# `sentence_transformers` pulls in the HF `datasets` package, which registers
# custom pyarrow extension types at import time. pyarrow's type registry is
# global to the process, so re-running this cell (or an autoreload-triggered
# re-import) tries to register the same type name twice and raises. It's a
# harmless duplicate registration, not a real error -- patch pyarrow to
# ignore duplicates before importing sentence_transformers/datasets so this
# cell is safe to re-run.
import pyarrow as pa
if not getattr(pa, "_dup_ext_type_patch_applied", False):
    _orig_register_extension_type = pa.register_extension_type

    def _safe_register_extension_type(ext_type):
        try:
            _orig_register_extension_type(ext_type)
        except pa.lib.ArrowKeyError:
            pass  # already registered earlier in this kernel session -- ignore

    pa.register_extension_type = _safe_register_extension_type
    pa._dup_ext_type_patch_applied = True

from sentence_transformers import SentenceTransformer

df = pd.read_parquet(PARQUET_PATH)
embeddings = np.stack(df["embedding"].to_numpy()).astype("float32")
device_level = df.drop_duplicates(subset="row_uid").copy()

embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # must match M2's embedding model
print(df.shape, embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(4171, 22) (4171, 384)


## 1b. Load the local LLM (4-bit, runs on your T4s)

`device_map="auto"` lets `accelerate` place layers across both T4s automatically if needed —
in practice a 7B model in 4-bit (~5GB) fits on a single T4 (16GB), so this mostly just works and
leaves your second GPU free for anything else you're running in parallel.

Swap `MODEL_ID` for `"meta-llama/Meta-Llama-3.1-8B-Instruct"` if you'd rather use Llama —
that one is gated on Hugging Face, so you'd need to accept the license on the model page and pass
`token="hf_..."` (your HF access token) into both `from_pretrained` calls below.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # open weights, no gating, strong instruction-following

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = Turing, no bfloat16 support -> use float16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.3,
    top_p=0.9,
    return_full_text=False,
)

print(model.hf_device_map)  # shows which GPU(s) each layer landed on


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 2. Metadata filter extraction

Pulls hard constraints (price, RAM, GPU brand, CPU tier, storage type) out of the raw query text with regex/keyword rules, and applies them to the device pool *before* vector search runs.


In [5]:
def extract_filters(query: str):
    q = query.lower()
    mask = pd.Series(True, index=device_level.index)
    applied = []

    m = re.search(r"under\s*\$?(\d+)", q) or re.search(r"less than\s*\$?(\d+)", q) or re.search(r"budget.*?\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] < price
        applied.append(f"price_usd < {price}")

    m = re.search(r"over\s*\$?(\d+)", q) or re.search(r"above\s*\$?(\d+)", q)
    if m:
        price = float(m.group(1))
        mask &= device_level["price_usd"] > price
        applied.append(f"price_usd > {price}")

    m = re.search(r"(\d+)\s*gb\s*ram", q) or re.search(r"at least\s*(\d+)\s*gb", q)
    if m:
        ram = float(m.group(1))
        mask &= device_level["ram_gb"] >= ram
        applied.append(f"ram_gb >= {ram}")

    if any(w in q for w in ["nvidia", "rtx", "geforce", "gaming"]):
        mask &= device_level["gpu"].str.contains("NVIDIA|RTX|GeForce", case=False, na=False)
        applied.append("gpu contains NVIDIA/RTX/GeForce")

    if "amd" in q or "ryzen" in q:
        mask &= device_level["cpu"].str.contains("Ryzen|AMD", case=False, na=False)
        applied.append("cpu contains Ryzen/AMD")

    for cpu_kw in ["i9", "i7", "i5", "i3"]:
        if cpu_kw in q:
            mask &= device_level["cpu"].str.contains(cpu_kw, case=False, na=False)
            applied.append(f"cpu contains {cpu_kw}")

    if "ssd" in q or "nvme" in q:
        mask &= device_level["storage"].str.contains("SSD|PCIe|NVMe", case=False, na=False)
        applied.append("storage is SSD/PCIe/NVMe")

    return mask, applied

# quick sanity check
mask, applied = extract_filters("gaming laptop with NVIDIA graphics under $1200")
print("filters applied:", applied)
print("candidates:", mask.sum())


filters applied: ['price_usd < 1200.0', 'gpu contains NVIDIA/RTX/GeForce']
candidates: 124


## 3. Retrieval — filter first, then FAISS search within the filtered pool


In [6]:
from abc import ABC, abstractmethod
import numpy as np

class ANNIndex(ABC):
    @abstractmethod
    def build(self, embeddings: np.ndarray):
        pass

    @abstractmethod
    def search(self, query_vec: np.ndarray, k: int):
        pass

In [7]:
class FlatL2(ANNIndex):
    def __init__(self):
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]

        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)

In [8]:
class IVFFlat(ANNIndex):
    def __init__(self, nlist=100, nprobe=10):
        self.nlist = nlist
        self.nprobe = nprobe
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]

        # Cannot have more clusters than training vectors
        nlist = min(self.nlist, len(embeddings))
        nlist = max(1, nlist)

        quantizer = faiss.IndexFlatL2(dim)

        self.index = faiss.IndexIVFFlat(
            quantizer,
            dim,
            nlist,
            faiss.METRIC_L2
        )

        # IVF requires training
        self.index.train(embeddings)

        self.index.add(embeddings)

        self.index.nprobe = min(self.nprobe, nlist)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)

In [9]:
class PQ(ANNIndex):
    def __init__(self, m=8, bits=8):
        self.m = m
        self.bits = bits
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]

        if dim % self.m != 0:
            raise ValueError(
                f"Embedding dimension ({dim}) must be divisible by m ({self.m})"
            )

        self.index = faiss.IndexPQ(
            dim,
            self.m,
            self.bits
        )

        # PQ requires training
        self.index.train(embeddings)

        self.index.add(embeddings)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)

In [10]:
def _safe_nlist(requested_nlist: int, n_vectors: int) -> int:
    max_nlist = max(1, n_vectors // 39)

    return min(requested_nlist, max_nlist)

class IVFPQ(ANNIndex):
    def __init__(
        self,
        nlist=100,
        nprobe=10,
        m=8,
        bits=8
    ):
        self.nlist = nlist
        self.nprobe = nprobe
        self.m = m
        self.bits = bits
        self.index = None


    def build(self, embeddings):
        dim = embeddings.shape[1]

        if dim % self.m != 0:
            raise ValueError(
                f"Embedding dimension ({dim}) must be divisible by m ({self.m})"
            )

        nlist = min(self.nlist, len(embeddings))
        nlist = max(1, nlist)
        actual_nlist = _safe_nlist(self.nlist, len(embeddings))

        quantizer = faiss.IndexFlatL2(dim)

        self.index = faiss.IndexIVFPQ(
            quantizer,
            dim,
            nlist,
            self.m,
            self.bits
        )

        # IVFPQ requires training
        self.index.train(embeddings)

        self.index.add(embeddings)

        self.index.nprobe = min(self.nprobe, nlist)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)

In [11]:
class HNSW(ANNIndex):
    def __init__(
        self,
        M=32,
        ef_construction=40,
        ef_search=16
    ):
        self.M = M
        self.ef_construction = ef_construction
        self.ef_search = ef_search
        self.index = None

    def build(self, embeddings):
        dim = embeddings.shape[1]

        self.index = faiss.IndexHNSWFlat(
            dim,
            self.M,
            faiss.METRIC_L2
        )

        self.index.hnsw.efConstruction = self.ef_construction
        self.index.hnsw.efSearch = self.ef_search

        # HNSW does not require training
        self.index.add(embeddings)

    def search(self, query_vec, k):
        return self.index.search(query_vec, k)

In [12]:
from enum import Enum

class IndexPreset(Enum):
    MAX = "max"
    EXTRA = "extra"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

In [13]:
print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

print("\nDataFrame shape:", df.shape)

print("\nUnique row_uids:", df["row_uid"].nunique())

print("\nChunks per row_uid:")
print(df.groupby("row_uid").size().describe())

print("\nDevice-level rows:", len(device_level))

Number of embeddings: 4171
Embedding shape: (4171, 384)
Embedding dtype: float32

DataFrame shape: (4171, 22)

Unique row_uids: 1960

Chunks per row_uid:
count    1960.000000
mean        2.128061
std         1.450714
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max         9.000000
dtype: float64

Device-level rows: 1960


In [14]:
import numpy as np

print("Embedding dimension:", embeddings.shape[1])
print("Number of vectors:", embeddings.shape[0])

norms = np.linalg.norm(
    embeddings.astype("float32"),
    axis=1
)

print("\nEmbedding norm:")
print("min :", norms.min())
print("max :", norms.max())
print("mean:", norms.mean())
print("std :", norms.std())

Embedding dimension: 384
Number of vectors: 4171

Embedding norm:
min : 0.9999999
max : 1.0000001
mean: 1.0
std : 4.2403737e-08


In [15]:
print(embed_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


```
Vectors:       2,643
Dimension:       384
Unique docs:   1,503
Normalized:      Yes
```

| Preset | IVFFlat `nlist` | `nprobe` | PQ `m` | PQ `bits` | HNSW `M` | `efSearch` |
| ------ | --------------: | -------: | -----: | --------: | -------: | ---------: |
| Max    |              32 |       32 |     16 |         4 |       32 |        128 |
| Extra  |              24 |       16 |     16 |         4 |       24 |         64 |
| High   |              16 |        8 |      8 |         4 |       16 |         32 |
| Medium |               8 |        4 |      8 |         4 |       12 |         16 |
| Low    |               4 |        2 |      4 |         4 |        8 |          8 |

In [16]:
def create_index(preset: IndexPreset, index_type: str = "ivfpq") -> ANNIndex:

    if isinstance(preset, str):
        preset = IndexPreset(preset.lower())

    index_type = index_type.lower()

    # ========================================================
    # MAX
    # ========================================================
    if preset == IndexPreset.MAX:

        if index_type == "flat":
            return FlatL2()

        if index_type == "ivfflat":
            return IVFFlat(nlist=32, nprobe=32)

        if index_type == "pq":
            return PQ(m=16, bits=4)

        if index_type == "ivfpq":
            return IVFPQ(nlist=32, nprobe=32, m=16, bits=4)

        if index_type == "hnsw":
            return HNSW(M=32, ef_construction=200, ef_search=128)

    # ========================================================
    # EXTRA
    # ========================================================
    elif preset == IndexPreset.EXTRA:

        if index_type == "flat":
            return FlatL2()

        if index_type == "ivfflat":
            return IVFFlat(nlist=24, nprobe=16)

        if index_type == "pq":
            return PQ(m=16, bits=4)

        if index_type == "ivfpq":
            return IVFPQ(nlist=24, nprobe=16, m=16, bits=4)

        if index_type == "hnsw":
            return HNSW(M=24, ef_construction=150, ef_search=64)

    # ========================================================
    # HIGH
    # ========================================================
    elif preset == IndexPreset.HIGH:

        if index_type == "flat":
            return FlatL2()

        if index_type == "ivfflat":
            return IVFFlat(nlist=16, nprobe=8)

        if index_type == "pq":
            return PQ(m=8, bits=4)

        if index_type == "ivfpq":
            return IVFPQ(nlist=16, nprobe=8, m=8, bits=4)

        if index_type == "hnsw":
            return HNSW(M=16, ef_construction=100, ef_search=32)

    # ========================================================
    # MEDIUM
    # ========================================================
    elif preset == IndexPreset.MEDIUM:

        if index_type == "flat":
            return FlatL2()

        if index_type == "ivfflat":
            return IVFFlat(nlist=8, nprobe=4)

        if index_type == "pq":
            return PQ(m=8, bits=4)

        if index_type == "ivfpq":
            return IVFPQ(nlist=8, nprobe=4, m=8, bits=4)

        if index_type == "hnsw":
            return HNSW(M=12, ef_construction=80, ef_search=16)

    # ========================================================
    # LOW
    # ========================================================
    elif preset == IndexPreset.LOW:

        if index_type == "flat":
            return FlatL2()

        if index_type == "ivfflat":
            return IVFFlat(nlist=4, nprobe=2)

        if index_type == "pq":
            return PQ(m=4, bits=4)

        if index_type == "ivfpq":
            return IVFPQ(nlist=4, nprobe=2, m=4, bits=4)

        if index_type == "hnsw":
            return HNSW(M=8, ef_construction=40, ef_search=8)

    raise ValueError(
        f"Unsupported preset/index combination: " f"{preset.value}/{index_type}"
    )


In [17]:
indexes = {
    "ivfpq": {},
    "ivfflat": {},
    "pq": {},
    "hnsw": {},
    "flat": {},
}

In [18]:
def retrieve(
    query: str,
    k: int = 5,
    index_type: str = "ivfpq",
    preset: str = "medium",
    candidate_k: int = 100,
):
    index_type = index_type.lower()
    preset = preset.lower()

    # --------------------------------------------------------
    # 1. Extract metadata filters
    # --------------------------------------------------------
    mask, applied_filters = extract_filters(query)

    candidate_uids = set(
        device_level.loc[mask, "row_uid"]
    )

    # --------------------------------------------------------
    # 2. Same fallback behavior as your old retrieve()
    # --------------------------------------------------------
    if len(candidate_uids) < k:
        candidate_uids = set(
            device_level["row_uid"]
        )

        applied_filters = applied_filters + [
            "(filters relaxed — too few matches)"
        ]

    # --------------------------------------------------------
    # 3. Get cached / create index
    # --------------------------------------------------------
    if index_type not in indexes:
        raise ValueError(
            f"Unknown index type: {index_type}. "
            f"Available: {list(indexes.keys())}"
        )

    if preset in indexes[index_type]:

        index = indexes[index_type][preset]

        print(
            f"Using cached index: "
            f"{index_type}/{preset}"
        )

    else:

        print(
            f"Creating index: "
            f"{index_type}/{preset}"
        )

        index = create_index(
            preset=preset,
            index_type=index_type
        )

        index.build(
            embeddings.astype("float32")
        )

        indexes[index_type][preset] = index

    # --------------------------------------------------------
    # 4. Encode query
    # --------------------------------------------------------
    query_vec = embed_model.encode(
        [query]
    ).astype("float32")

    # --------------------------------------------------------
    # 5. Search MORE candidates than k
    # --------------------------------------------------------
    search_k = min(
        candidate_k,
        len(df)
    )

    distances, global_idx = index.search(
        query_vec,
        search_k
    )

    global_idx = global_idx[0]

    # Remove invalid FAISS indices
    global_idx = global_idx[
        global_idx >= 0
    ]

    # --------------------------------------------------------
    # 6. Apply metadata filtering to FAISS results
    # --------------------------------------------------------
    filtered_idx = []

    for idx in global_idx:

        row_uid = df.iloc[idx]["row_uid"]

        if row_uid in candidate_uids:
            filtered_idx.append(idx)

        if len(filtered_idx) >= k:
            break

    # --------------------------------------------------------
    # 7. If not enough filtered results, relax filters
    # --------------------------------------------------------
    if len(filtered_idx) < k:

        applied_filters = applied_filters + [
            "(filters relaxed — insufficient vector matches)"
        ]

        filtered_idx = global_idx[:k]

    # --------------------------------------------------------
    # 8. Build result dataframe
    # --------------------------------------------------------
    results = df.iloc[filtered_idx][
        [
            "title",
            "price_usd",
            "cpu",
            "ram_gb",
            "storage",
            "gpu",
            "display",
            "battery",
            "chunk_text",
            "lineage",
        ]
    ]

    return results, applied_filters

In [19]:
for index_type in ["ivfpq", "ivfflat", "pq", "hnsw", "flat"]:
    for preset in ["max", "extra", "high", "medium", "low"]:
        index = create_index(preset=preset, index_type=index_type)
        index.build(embeddings.astype("float32"))
        indexes[index_type][preset] = index
        
indexes

{'ivfpq': {'max': <__main__.IVFPQ at 0x7bf69c851fa0>,
  'extra': <__main__.IVFPQ at 0x7bf69c851cd0>,
  'high': <__main__.IVFPQ at 0x7bf69c662e40>,
  'medium': <__main__.IVFPQ at 0x7bf69c4c9be0>,
  'low': <__main__.IVFPQ at 0x7bf69c4cbb90>},
 'ivfflat': {'max': <__main__.IVFFlat at 0x7bf69c4c92e0>,
  'extra': <__main__.IVFFlat at 0x7bf69c4c9a30>,
  'high': <__main__.IVFFlat at 0x7bf69c4c9430>,
  'medium': <__main__.IVFFlat at 0x7bf69c4cbec0>,
  'low': <__main__.IVFFlat at 0x7bf69c4c9cd0>},
 'pq': {'max': <__main__.PQ at 0x7bf69c4caab0>,
  'extra': <__main__.PQ at 0x7bf69c4cb6e0>,
  'high': <__main__.PQ at 0x7bf69c4c9820>,
  'medium': <__main__.PQ at 0x7bf69c4cb800>,
  'low': <__main__.PQ at 0x7bf69c4c9e80>},
 'hnsw': {'max': <__main__.HNSW at 0x7bf69c4c9880>,
  'extra': <__main__.HNSW at 0x7bf69c4cb5f0>,
  'high': <__main__.HNSW at 0x7bf69c4cbb60>,
  'medium': <__main__.HNSW at 0x7bf69c4ca6c0>,
  'low': <__main__.HNSW at 0x7bf69c4ca510>},
 'flat': {'max': <__main__.FlatL2 at 0x7bf69c4cb

In [20]:
res, applied_filters = retrieve(
    "gaming laptop with NVIDIA graphics under $1200",
    index_type = "flat",
    preset = "max"
)
res

Using cached index: flat/max


,title,price_usd,cpu,ram_gb,storage,gpu,display,battery,chunk_text,lineage
314,Lenovo Legion Y520-15IKBN,1148.85,Intel Core i7 7700HQ 2.8GHz,4.0,1TB HDD,Nvidia GeForce GTX 1050,"15.6"" IPS Panel Full HD 1920x1080",54.3 Wh,Product: Lenovo Legion Y520-15IKBN (Gaming) Pr...,chunk:ae8046037e8b0814bfa457dfebb443c1_0 <- de...
1368,"Acer 16"" Nitro V Gaming Laptop",1149.99,Intel Core 7 240H,32.0,512 GB,NVIDIA GeForce RTX 5060 with 8 GB GDDR7 VRAM,"16""",76 Wh,V Gaming Laptop . Equipped with an Intel Core ...,chunk:fd111fd79863cfbe5141d051e1dfcc97_3 <- de...
3458,Lenovo Legion Y520-15IKBN,941.85,Intel Core i5 7300HQ 2.5GHz,8.0,1TB HDD,Nvidia GeForce GTX 1050,"15.6"" IPS Panel Full HD 1920x1080",43.9 Wh,Product: Lenovo Legion Y520-15IKBN (Gaming) Pr...,chunk:5e9ecf305f360f79a2bb07918d1b741c_0 <- de...
1257,Lenovo Legion Y520-15IKBN,999.35,Intel Core i7 7700HQ 2.8GHz,8.0,256GB SSD,Nvidia GeForce GTX 1050M,"15.6"" IPS Panel Full HD 1920x1080",47.5 Wh,Product: Lenovo Legion Y520-15IKBN (Gaming) Pr...,chunk:22df04e08707758bcc7ac32e88f544f1_0 <- de...
4005,Asus Rog GL552VW-DM201T,1045.35,Intel Core i7 6700HQ 2.6GHz,8.0,256GB SSD + 1TB HDD,Nvidia GeForce GTX 960M,"15.6"" IPS Panel Full HD 1920x1080",46.7 Wh,Product: Asus Rog GL552VW-DM201T (Gaming) Pric...,chunk:fba20a090e4b8cb2ddac280bdd2a0939_0 <- de...


## 4. Generation — local LLM produces a grounded, justified recommendation from retrieved context only

Same grounded-prompt design as the Groq version (only use the listed specs, don't invent devices),
built through the model's chat template and run through the local pipeline instead of an API call.


In [21]:
def generate_recommendation(query: str, retrieved_df: pd.DataFrame):
    context_blocks = []
    for i, row in retrieved_df.iterrows():
        context_blocks.append(
            f"- {row['title']} | ${row['price_usd']:.2f} | {row['cpu']} | {row['ram_gb']}GB RAM | "
            f"{row['storage']} | {row['gpu']} | {row['display']}"
        )
    context = "\n".join(context_blocks)

    user_prompt = (
        "You are a laptop recommendation assistant. Base your answer ONLY on the devices listed "
        "below — do not invent specs or devices not present here.\n\n"
        f"User request: {query}\n\n"
        f"Retrieved candidate devices:\n{context}\n\n"
        "Give a short recommendation: pick the best match (or top 2), and justify the choice "
        "using only the specs shown above."
    )

    messages = [{"role": "user", "content": user_prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    output = llm_pipe(prompt_text)
    return output[0]["generated_text"].strip()

answer = generate_recommendation("gaming laptop with NVIDIA graphics under $1200", res)
print(answer)

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the user's requirement for a gaming laptop with NVIDIA graphics under $1200, I recommend the following options:

1. **Lenovo Legion Y520-15IKBN | $1148.85 | Intel Core i7 7700HQ 2.8GHz | 4.0GB RAM | 1TB HDD | Nvidia GeForce GTX 1050 | 15.6" IPS Panel Full HD 1920x1080**

2. **Acer 16" Nitro V Gaming Laptop | $1149.99 | Intel Core 7 240H | 32.0GB RAM | 512 GB | NVIDIA GeForce RTX 5060 with 8 GB GDDR7 VRAM | 16"**

### Justification:

**First Option (Lenovo Legion Y520-15IKBN):**
- **Processor:** Intel Core i7 7700HQ (2.8GHz) - This is a powerful processor suitable for gaming.
- **RAM:** 4.0GB - While this is less than the recommended minimum of 8GB, it's still sufficient for basic gaming needs.
- **Storage:** 1TB HDD - Provides ample storage space.
- **Graphics:** Nvidia GeForce GTX 1050 - A mid-range GPU that offers good performance for most games at medium to high settings.
- **Display:** 15.6" IPS Panel Full HD 1920x1080 - Offers a good balance between screen size and resolu

## 5. Full pipeline wrapper — end to end query -> answer


In [22]:
def recommend(query: str, k: int, index_type: str, preset: str):
    retrieved, applied_filters = retrieve(query, k, index_type, preset)

    answer = generate_recommendation(query, retrieved)

    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": retrieved
            .drop(columns=["lineage"])
            .to_dict(orient="records"),
        "lineage": retrieved["lineage"].tolist(),
        "recommendation": answer,
    }

result = recommend(
    "lightweight ultrabook under $800 for a student",
    5,
    "ivfpq",
    "high"
)

print(json.dumps(result, indent=2))

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: ivfpq/high
{
  "query": "lightweight ultrabook under $800 for a student",
  "filters_applied": [
    "price_usd < 800.0"
  ],
  "retrieved_context": [
    {
      "title": "ASUS VivoBook 14 Laptop, 14\" FHD, AMD Ryzen 7-3700U, AMD Radeon RX Vega 10 Graphics, 8 GB DDR4 RAM, 512 GB PCIe SSD, Backlit KB, Fingerprint, Windows 10 Home, Slate Grey, F412DA-NH77",
      "price_usd": 499.0,
      "cpu": "AMD Ryzen 7 3000 Series",
      "ram_gb": 8.0,
      "storage": "512GB",
      "gpu": "AMD Radeon RX Vega 10",
      "display": "14.0\"",
      "battery": "37 WHrs, 2S1P, 2-cell Li-ion",
      "chunk_text": "Asus for a good solid 14\" notebook without going full ultrabook. No issues, runs fast and does all my IT chores, web, internet, streaming video, email, and anything I throw at it right now. | I recommend it for any student or anyone who look for a good battery life and a light laptop with light use like word, PowerPoint , Microsoft teams."
    },
    {
      "title": "A

## 5a. Grounding-verification helpers

Turns each applied filter string (e.g. `"price_usd < 600"`) into a checkable
`(column, operator, threshold)` triple, then evaluates it against a row's actual value.

In [23]:
def parse_filter_condition(filter_str: str):
    """Turn an applied-filter string like 'price_usd < 600' into (column, operator, threshold).
    Returns None for non-numeric filters (e.g. 'gpu contains NVIDIA/RTX/GeForce') since those
    aren't checked as inequalities."""
    m = re.match(r"(\w+)\s*(>=|<=|>|<|==)\s*([\d.]+)", filter_str.strip())
    if not m:
        return None
    col, op, threshold = m.group(1), m.group(2), float(m.group(3))
    return (col, op, threshold)


def check_condition(actual, op, threshold):
    """Evaluate actual <op> threshold for the four comparison operators extract_filters can produce."""
    if op == "<":
        return actual < threshold
    if op == "<=":
        return actual <= threshold
    if op == ">":
        return actual > threshold
    if op == ">=":
        return actual >= threshold
    if op == "==":
        return actual == threshold
    return True  # unknown operator -> don't flag it

# sanity check
print(parse_filter_condition("price_usd < 600"))          # ("price_usd", "<", 600.0)
print(parse_filter_condition("gpu contains NVIDIA/RTX"))  # None (non-numeric, skipped)
print(check_condition(479.99, "<", 600.0))                 # True


('price_usd', '<', 600.0)
None
True


## 5b. Grounding verification (catches LLM factual errors about its own retrieved context)

Local 7B models are, if anything, *more* prone to this kind of slip than a larger hosted model, so
this check matters at least as much here as it did with Groq's Llama-3.1-8B.


In [24]:
def verify_grounding(query: str, retrieved_df: pd.DataFrame, applied_filters: list, answer_text: str):
    """Cross-check the LLM's answer against the actual retrieved data.
    Returns (verified_facts, warnings) — warnings flag likely hallucinated/incorrect claims."""
    numeric_filters = [parse_filter_condition(f) for f in applied_filters]
    numeric_filters = [f for f in numeric_filters if f is not None]

    verified_facts = []
    warnings = []
    answer_lower = answer_text.lower()

    for _, row in retrieved_df.iterrows():
        title = row["title"]
        title_snippet = title[:25].lower()
        if title_snippet not in answer_lower:
            continue

        fact_line = f"{title[:60]}... -> price ${row['price_usd']:.2f}"
        checks = []
        for col, op, threshold in numeric_filters:
            if col not in row or pd.isna(row[col]):
                continue
            actual = row[col]
            satisfies = check_condition(actual, op, threshold)
            checks.append(f"{col} {op} {threshold} -> {'PASS' if satisfies else 'FAIL'} (actual={actual})")
            if not satisfies:
                warnings.append(
                    f"MISMATCH: '{title[:60]}...' does NOT satisfy {col} {op} {threshold} "
                    f"(actual {col}={actual}), but was still recommended."
                )
        verified_facts.append(fact_line + " | " + "; ".join(checks) if checks else fact_line)

    contradiction_patterns = [
        r"above\s+(?:the\s+|a\s+|our\s+)?budget",
        r"over\s+(?:the\s+|a\s+|our\s+)?budget",
        r"exceed\w*\s+(?:the\s+|a\s+|our\s+)?budget",
        r"not\s+(?:strictly\s+)?under\s+(?:the\s+|a\s+|our\s+)?budget",
        r"too\s+expensive",
        r"outside\s+(?:the\s+|a\s+|our\s+)?budget",
        r"beyond\s+(?:the\s+|a\s+|our\s+)?budget",
    ]
    negation_words = ["without", "not", "no", "doesn't", "does not", "isn't", "is not", "won't", "never"]

    sentences = re.split(r'(?<=[.!?])\s+', answer_text)
    for sentence in sentences:
        sentence_lower = sentence.lower()
        for pattern in contradiction_patterns:
            match = re.search(pattern, sentence_lower)
            if match:
                preceding_text = sentence_lower[:match.start()]
                preceding_words = preceding_text.split()[-4:]
                if any(neg in " ".join(preceding_words) for neg in negation_words):
                    continue
                warnings.append(
                    f"POSSIBLE HALLUCINATION: sentence claims a budget/threshold violation "
                    f"— verify manually: \"{sentence.strip()}\""
                )
                break

    return verified_facts, warnings


## 5c. `recommend_verified()` — wraps retrieval + local generation + grounding check


In [25]:
def recommend_verified(query: str, index_type, preset, k: int = 5):
    retrieved, applied_filters = retrieve(query, k, index_type, preset)
    answer = generate_recommendation(query, retrieved)
    verified_facts, warnings = verify_grounding(query, retrieved, applied_filters, answer)

    return {
        "query": query,
        "filters_applied": applied_filters,
        "retrieved_context": retrieved.drop(columns=["lineage"]).to_dict(orient="records"),
        "lineage": retrieved["lineage"].tolist(),
        "recommendation": answer,
        "grounding_check": {
            "verified_facts": verified_facts,
            "warnings": warnings,
            "passed": len(warnings) == 0,
        },
    }

result = recommend_verified("gaming laptop with NVIDIA graphics under $1200", "flat", "max")
print(json.dumps(result, indent=2))


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: flat/max
{
  "query": "gaming laptop with NVIDIA graphics under $1200",
  "filters_applied": [
    "price_usd < 1200.0",
    "gpu contains NVIDIA/RTX/GeForce"
  ],
  "retrieved_context": [
    {
      "title": "Lenovo Legion Y520-15IKBN",
      "price_usd": 1148.85,
      "cpu": "Intel Core i7 7700HQ 2.8GHz",
      "ram_gb": 4.0,
      "storage": "1TB HDD",
      "gpu": "Nvidia GeForce GTX 1050",
      "display": "15.6\" IPS Panel Full HD 1920x1080",
      "battery": "54.3 Wh",
      "chunk_text": "Product: Lenovo Legion Y520-15IKBN (Gaming) Price: EUR 999.00 Specifications: CPU: Intel Core i7 7700HQ 2.8GHz; RAM: 4GB; Storage: 1TB HDD; GPU: Nvidia GeForce GTX 1050; Display: 15.6\" IPS Panel Full HD 1920x1080; OS: Windows 10; Weight: 2.4kg"
    },
    {
      "title": "Acer 16\" Nitro V Gaming Laptop",
      "price_usd": 1149.99,
      "cpu": "Intel Core 7 240H",
      "ram_gb": 32.0,
      "storage": "512 GB",
      "gpu": "NVIDIA GeForce RTX 5060 with 8 GB GDDR7 VR

## 7. Retrieval evaluation — precision@k / recall@k

Runs every labeled query through the real `retrieve()` (filters + FAISS), scored against
`eval_queries.csv`. As flagged before: that file's queries are auto-generated from each row's own
title+CPU, so they're not truly "hand-labeled" per the spec and don't exercise the filter logic —
worth hand-writing/relabeling a subset before you report these numbers in the M4 report.


In [26]:
eval_df = pd.read_csv(EVAL_CSV_PATH)
print(eval_df.shape)
print("columns:", eval_df.columns.tolist())
assert len(eval_df) >= 30, "spec requires at least 30 labeled queries"
eval_df.head()


(30, 3)
columns: ['query', 'n_relevant', 'relevant_row_uids']


,query,n_relevant,relevant_row_uids
0,cheap laptop under $400 for basic tasks,228,071b7fb7929d290695ab003ff50d535a|f49c5a71748c2...
1,gaming laptop with NVIDIA RTX under $1200,3,d7d848bc633aa5d3044e30bb75305339|1165991b794a8...
2,lightweight ultrabook under 3 lbs,138,a06dbf23a83024459f3318ba59a0c9c6|ff18ce511818a...
3,laptop with 32GB RAM and 1TB SSD,113,558389f6752bb2a5f024a76c8b26a6ed|04afd2024216e...
4,AMD Ryzen laptop under $600,35,34950ed249e57741546384aef2b1f675|018c8a2835d81...


In [27]:
def evaluate_pipeline_retrieval(eval_df: pd.DataFrame, k: int = 5):
    """Run every labeled query through the real M4 retrieve() (filters + FAISS), and score
    precision@k / recall@k against the *set* of known-relevant row_uids per query.

    eval_queries.csv format: one row per query, with a "relevant_row_uids" column holding
    a "|"-separated list of every relevant device's row_uid (there can be many relevant
    devices per query, e.g. every laptop under a price threshold) -- NOT a single uid.
    """
    rows = []
    for _, row in eval_df.iterrows():
        query = row["query"]
        relevant_uids = set(str(row["relevant_row_uids"]).split("|"))
        n_relevant = len(relevant_uids)

        retrieved, applied_filters = retrieve(query, k=k)
        merged = retrieved.merge(
            device_level[["row_uid", "title", "price_usd"]],
            on=["title", "price_usd"], how="left"
        )
        retrieved_uids = set(merged["row_uid"].dropna())

        n_retrieved = len(retrieved_uids)
        n_hits = len(retrieved_uids & relevant_uids)

        rows.append({
            "query": query,
            "n_relevant": n_relevant,
            "n_retrieved": n_retrieved,
            "n_filters_applied": len([f for f in applied_filters if "relaxed" not in f]),
            "n_hits": n_hits,
            "hit": n_hits > 0,
            "precision_at_k": (n_hits / n_retrieved) if n_retrieved > 0 else 0.0,
            "recall_at_k": (n_hits / n_relevant) if n_relevant > 0 else 0.0,
        })

    return pd.DataFrame(rows)

k = 5
retrieval_eval_df = evaluate_pipeline_retrieval(eval_df, k=k)
retrieval_eval_df.to_csv(os.path.join(OUT_DIR, "m4_retrieval_evaluation.csv"), index=False)

print(f"=== M4 pipeline retrieval evaluation (k={k}, n={len(retrieval_eval_df)} queries) ===")
print(f"Mean precision@{k}: {retrieval_eval_df['precision_at_k'].mean():.3f}")
print(f"Mean recall@{k}:    {retrieval_eval_df['recall_at_k'].mean():.3f}")
print(f"Hit rate:           {retrieval_eval_df['hit'].mean():.3f}")
retrieval_eval_df.head(10)


Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
Using cached index: ivfpq/medium
=== M4 pip

,query,n_relevant,n_retrieved,n_filters_applied,n_hits,hit,precision_at_k,recall_at_k
0,cheap laptop under $400 for basic tasks,228,5,1,5,True,1.00,0.021930
1,gaming laptop with NVIDIA RTX under $1200,3,4,2,1,True,0.25,0.333333
2,lightweight ultrabook under 3 lbs,138,5,1,0,False,0.00,0.000000
3,laptop with 32GB RAM and 1TB SSD,113,5,2,4,True,0.80,0.035398
4,AMD Ryzen laptop under $600,35,4,2,4,True,1.00,0.114286
5,Intel Core i7 laptop with NVIDIA graphics,202,5,2,2,True,0.40,0.009901
6,Intel Core i5 laptop under $700,140,5,2,0,False,0.00,0.000000
7,laptop with a 1TB SSD under $900,27,5,2,3,True,0.60,0.111111
8,touchscreen 2-in-1 convertible laptop,78,5,0,0,False,0.00,0.000000
9,premium laptop over $1500 with 32GB RAM,66,4,2,0,False,0.00,0.000000


## 8. Qualitative assessment of generated answers


In [28]:
qualitative_queries = [
    "gaming laptop with NVIDIA graphics under $1200",
    "AMD Ryzen laptop under $600",
    "lightweight ultrabook for a college student",
    "laptop with at least 32GB RAM and 1TB SSD storage",
    "cheapest laptop with an i9 processor",
]

qualitative_results = []
for q in qualitative_queries:
    result = recommend_verified(q, "flat", "max", k=5)
    qualitative_results.append(result)
    print("=" * 100)
    print("QUERY:", q)
    print("FILTERS:", result["filters_applied"])
    print("N CANDIDATES RETURNED:", len(result["retrieved_context"]))
    print("-" * 100)
    print("ANSWER:\n", result["recommendation"])
    print("-" * 100)
    print("GROUNDING CHECK PASSED:", result["grounding_check"]["passed"])
    if result["grounding_check"]["warnings"]:
        print("WARNINGS:")
        for w in result["grounding_check"]["warnings"]:
            print(" -", w)
    print()

with open(os.path.join(OUT_DIR, "m4_qualitative_samples.json"), "w") as f:
    json.dump(qualitative_results, f, indent=2)


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: flat/max


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: gaming laptop with NVIDIA graphics under $1200
FILTERS: ['price_usd < 1200.0', 'gpu contains NVIDIA/RTX/GeForce']
N CANDIDATES RETURNED: 5
----------------------------------------------------------------------------------------------------
ANSWER:
 Based on the user's requirement for a gaming laptop with NVIDIA graphics under $1200, I recommend the following two options:

1. **Lenovo Legion Y520-15IKBN | $1148.85 | Intel Core i7 7700HQ 2.8GHz | 4.0GB RAM | 1TB HDD | Nvidia GeForce GTX 1050 | 15.6" IPS Panel Full HD 1920x1080**
2. **Lenovo Legion Y520-15IKBN | $999.35 | Intel Core i7 7700HQ 2.8GHz | 8.0GB RAM | 256GB SSD | Nvidia GeForce GTX 1050M | 15.6" IPS Panel Full HD 1920x1080**

### Justification:

**First Option (Lenovo Legion Y520-15IKBN @ $1148.85):**
- **Processor:** Intel Core i7 7700HQ (2.8GHz) - This is a powerful processor that can handle most gaming tasks efficiently.
- **RAM:** 4.0GB - While this is the minimum requirement for many games, it might be slightly lim

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: AMD Ryzen laptop under $600
FILTERS: ['price_usd < 600.0', 'cpu contains Ryzen/AMD']
N CANDIDATES RETURNED: 5
----------------------------------------------------------------------------------------------------
ANSWER:
 Based on the user's requirement for an AMD Ryzen laptop under $600, I recommend the following two options:

1. **Lenovo Notebooks IdeaPad AMD Ryzen 7 5700U (1.80GHz) 12GB Memory 512 GB SSD AMD Radeon Graphics 15.6" Touchscreen Windows 11 Home** - $549.00
2. **Lenovo IdeaPad 1 15.6" FHD Laptop, AMD Ryzen 5 7520U, 8GB RAM, 512GB SSD, Windows 11 Home, Blue** - $479.00

### Justification:

- **Performance**: The Lenovo IdeaPad 1 with an AMD Ryzen 5 7520U offers a balanced performance at a lower price point compared to the Lenovo Notebooks IdeaPad with an AMD Ryzen 7 5700U. However, the Lenovo Notebooks IdeaPad has more RAM (12GB vs. 8GB) which can be beneficial for multitasking and running resource-intensive applications.
  
- **Price**: Both laptops fall within the 

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: lightweight ultrabook for a college student
FILTERS: []
N CANDIDATES RETURNED: 5
----------------------------------------------------------------------------------------------------
ANSWER:
 Based on the user's requirement for a lightweight ultrabook suitable for a college student, I recommend the **ASUS VivoBook 14 Laptop** (F412DA-NH77).

### Justification:

1. **Weight and Portability**: The ASUS VivoBook 14 is a 14-inch ultrabook, which is generally considered more portable than the 13.3-inch laptops like the Toshiba Portege models. This makes it easier for a college student to carry around between classes and on-the-go.

2. **Performance**: It comes with an AMD Ryzen 7-3700U processor, which offers strong performance for multitasking, running applications, and handling coursework. The AMD Radeon RX Vega 10 graphics provide decent performance for basic gaming or video playback.

3. **Storage and Memory**: The device has 8 GB of DDR4 RAM and a 512 GB PCIe SSD, which ensures s

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUERY: laptop with at least 32GB RAM and 1TB SSD storage
FILTERS: ['ram_gb >= 32.0', 'storage is SSD/PCIe/NVMe']
N CANDIDATES RETURNED: 5
----------------------------------------------------------------------------------------------------
ANSWER:
 Based on the user's requirements for a laptop with at least 32GB RAM and 1TB SSD storage, the best matches from the provided list are:

1. **LG Gram Copilot+ PC Laptop Intel Core Ultra 7 258V 32GB Memory 2 TB M.2 SSD 17.0" Touchscreen Windows 11 Home (17Z90TL-H.AUB7U1)** - $1299.99
   - **Specs**: 32.0GB RAM, 2 TB M.2 Dual SSD Gen4 NVMe
   - **Justification**: This model meets the user's requirements with 32GB of RAM and 2TB of SSD storage, which exceeds the minimum requirement of 1TB. It also offers a larger screen size of 17 inches compared to the other options that meet the minimum requirements.

2. **Dell Plus 16" Touchscreen Laptop - Intel Core Ultra 9 288V - 1920 x 1920 - Windows 11 Home - 32GB RAM - 1TB SSD** - $1311.99
   - **Specs**:

## 9. Bonus — Semantic caching layer (stretch goal, spec section 4)

Caches full `recommend_verified()` responses (retrieval + local LLM generation + grounding check)
keyed by query *meaning*, not exact text — so a paraphrase of a recent query ("cheap AMD laptop"
vs "budget AMD Ryzen laptop") skips straight to a cached answer instead of re-running embedding +
FAISS + a full GPU generation pass.

This matters at least as much with a local model as it did with Groq: a 7B model on a T4 still
takes a few seconds per generation (no API round-trip, but token-by-token decoding isn't free),
so repeated/paraphrased queries are exactly where caching pays off.

Caches the **verified** response (not the raw uncached one), so a cache hit still carries the
grounding-check result — you're not skipping the hallucination check, just skipping the expensive
retrieval + generation work that produced it the first time.


In [29]:
import numpy as np

class SemanticCache:
    def __init__(self, similarity_threshold: float = 0.92, max_size: int = 200):
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.embeddings = []   # list of normalized query embeddings
        self.entries = []      # list of (original_query, response_dict)

    def _normalize(self, vec):
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def lookup(self, query_embedding: np.ndarray):
        if not self.embeddings:
            return None, None
        q = self._normalize(query_embedding)
        sims = np.array([np.dot(q, e) for e in self.embeddings])
        best_idx = int(np.argmax(sims))
        best_sim = float(sims[best_idx])
        if best_sim >= self.threshold:
            matched_query, response = self.entries[best_idx]
            return response, {"matched_query": matched_query, "similarity": best_sim}
        return None, None

    def store(self, query: str, query_embedding: np.ndarray, response: dict):
        if len(self.embeddings) >= self.max_size:
            self.embeddings.pop(0)
            self.entries.pop(0)
        self.embeddings.append(self._normalize(query_embedding))
        self.entries.append((query, response))

    def __len__(self):
        return len(self.entries)


cache = SemanticCache(similarity_threshold=0.92)


def recommend_cached(query: str, index_type, preset, k: int = 5):
    """Semantic-cache wrapper around recommend_verified(). Embeds the query once, checks the
    cache, and only runs the full filter+retrieve+generate+verify pipeline on a miss."""
    query_embedding = embed_model.encode([query])[0].astype("float32")

    cached_response, match_info = cache.lookup(query_embedding)
    if cached_response is not None:
        result = dict(cached_response)
        result["cache_hit"] = True
        result["matched_query"] = match_info["matched_query"]
        result["similarity"] = round(match_info["similarity"], 4)
        return result

    result = recommend_verified(query, index_type, preset, k)
    result["cache_hit"] = False
    cache.store(query, query_embedding, result)
    return result


### 9a. Benchmark — latency and hit-rate, within a single run

**Design note:** an earlier version of this benchmark (see your `semantic_cache.ipynb`) compared a
cache-enabled run against a *separately run* uncached pass, and found the comparison was confounded
— Groq's API latency varied between runs for reasons unrelated to caching, inflating the apparent
speedup. Running local inference removes API-side jitter, but GPU latency still varies a little
run-to-run (thermal throttling, other processes on the GPU, etc.), so this version sidesteps the
problem entirely: it measures cache-hit vs. cache-miss latency **within the same run**, which is
apples-to-apples by construction — no cross-run comparison needed.

`benchmark_queries` mixes exact repeats, paraphrases (should hit the cache), and genuinely new
queries (should miss), so both branches get exercised.


In [30]:
import time

benchmark_queries = [
    "AMD Ryzen laptop under $600",
    "cheap AMD laptop under $600 dollars",                        # paraphrase of #1 -> should hit
    "gaming laptop with NVIDIA graphics under $1200",
    "budget laptop with AMD Ryzen processor, under 600 bucks",    # paraphrase of #1 -> should hit
    "lightweight ultrabook for travel",
    "AMD Ryzen laptop under $600",                                # exact repeat of #1 -> should hit
    "NVIDIA gaming laptop under $1200 budget",                    # paraphrase of #3 -> should hit
    "laptop with 32GB RAM and 1TB SSD",
]

cache = SemanticCache(similarity_threshold=0.92)  # fresh cache for a clean benchmark
timings = []
for q in benchmark_queries:
    start = time.perf_counter()
    result = recommend_cached(q, "flat", "max")
    elapsed = time.perf_counter() - start
    timings.append({
        "query": q,
        "cache_hit": result["cache_hit"],
        "latency_s": elapsed,
        "matched_query": result.get("matched_query"),
        "similarity": result.get("similarity"),
    })
    hit_str = f"HIT (matched: \"{result.get('matched_query')}\", sim={result.get('similarity')})" if result["cache_hit"] else "MISS"
    print(f"[{elapsed:.3f}s] {hit_str:70s} | {q}")

bench_df = pd.DataFrame(timings)
bench_df.to_csv(os.path.join(OUT_DIR, "semantic_cache_benchmark.csv"), index=False)

print()
print(f"Cache hit rate: {bench_df['cache_hit'].sum()}/{len(bench_df)} = {bench_df['cache_hit'].mean():.1%}")
avg_hit = bench_df[bench_df["cache_hit"]]["latency_s"].mean()
avg_miss = bench_df[~bench_df["cache_hit"]]["latency_s"].mean()
speedup = avg_miss / avg_hit if avg_hit > 0 else float("nan")
print(f"Avg latency on hits:   {avg_hit:.4f}s")
print(f"Avg latency on misses: {avg_miss:.4f}s")
print(f"Speedup on a cache hit vs. a fresh generation: {speedup:.1f}x")

summary = pd.DataFrame([{
    "hit_rate": bench_df["cache_hit"].mean(),
    "avg_latency_hit_s": avg_hit,
    "avg_latency_miss_s": avg_miss,
    "speedup_hit_vs_miss": speedup,
    "similarity_threshold": 0.92,
    "n_queries": len(bench_df),
}])
summary.to_csv(os.path.join(OUT_DIR, "semantic_cache_summary.csv"), index=False)
print()
print(summary.to_string(index=False))


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: flat/max


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[28.045s] MISS                                                                   | AMD Ryzen laptop under $600
Using cached index: flat/max


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[28.102s] MISS                                                                   | gaming laptop with NVIDIA graphics under $1200
Using cached index: flat/max


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[28.209s] MISS                                                                   | budget laptop with AMD Ryzen processor, under 600 bucks
Using cached index: flat/max


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[19.773s] MISS                                                                   | lightweight ultrabook for travel
[0.007s] HIT (matched: "AMD Ryzen laptop under $600", sim=1.0)                  | AMD Ryzen laptop under $600
[0.006s] HIT (matched: "gaming laptop with NVIDIA graphics under $1200", sim=0.9647) | NVIDIA gaming laptop under $1200 budget
Using cached index: flat/max
[28.172s] MISS                                                                   | laptop with 32GB RAM and 1TB SSD

Cache hit rate: 2/8 = 25.0%
Avg latency on hits:   0.0068s
Avg latency on misses: 25.4544s
Speedup on a cache hit vs. a fresh generation: 3770.1x

 hit_rate  avg_latency_hit_s  avg_latency_miss_s  speedup_hit_vs_miss  similarity_threshold  n_queries
     0.25           0.006752           25.454428          3770.075766                  0.92          8


### 9b. Threshold sensitivity

`0.92` was a design choice, not an arbitrary default — sweep a few thresholds to show it was
chosen deliberately. Too low risks false-positive hits (returning a wrong cached answer for a
meaningfully different query); too high barely ever hits at all.


In [31]:
threshold_results = []
for threshold in [0.80, 0.85, 0.90, 0.92, 0.95, 0.98]:
    test_cache = SemanticCache(similarity_threshold=threshold)
    hits = 0
    for q in benchmark_queries:
        query_embedding = embed_model.encode([q])[0].astype("float32")
        cached_response, match_info = test_cache.lookup(query_embedding)
        if cached_response is not None:
            hits += 1
        else:
            # simulate storing without a real generation call, to isolate the threshold effect
            test_cache.store(q, query_embedding, {"query": q, "recommendation": "placeholder"})
    threshold_results.append({"threshold": threshold, "hits": hits, "hit_rate": hits / len(benchmark_queries)})

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))
threshold_df.to_csv(os.path.join(OUT_DIR, "semantic_cache_threshold_sweep.csv"), index=False)


 threshold  hits  hit_rate
      0.80     3     0.375
      0.85     3     0.375
      0.90     3     0.375
      0.92     2     0.250
      0.95     2     0.250
      0.98     1     0.125


### 9c. Paraphrase similarity spot-check

Direct cosine similarities for the paraphrase pairs used above — useful supporting evidence in
the report for why `0.92` is a reasonable cutoff.


In [32]:
def cosine_sim(a, b):
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    return float(np.dot(a, b))

pairs_to_check = [
    ("AMD Ryzen laptop under $600", "cheap AMD laptop under $600 dollars"),
    ("AMD Ryzen laptop under $600", "budget laptop with AMD Ryzen processor, under 600 bucks"),
    ("gaming laptop with NVIDIA graphics under $1200", "NVIDIA gaming laptop under $1200 budget"),
]
similarity_results = []
for q1, q2 in pairs_to_check:
    e1 = embed_model.encode([q1])[0]
    e2 = embed_model.encode([q2])[0]
    sim = cosine_sim(e1, e2)
    similarity_results.append({"query_a": q1, "query_b": q2, "similarity": sim})
    print(f"{sim:.4f}  |  '{q1}'  vs  '{q2}'")

pd.DataFrame(similarity_results).to_csv(os.path.join(OUT_DIR, "semantic_cache_paraphrase_similarities.csv"), index=False)


0.7976  |  'AMD Ryzen laptop under $600'  vs  'cheap AMD laptop under $600 dollars'
0.9113  |  'AMD Ryzen laptop under $600'  vs  'budget laptop with AMD Ryzen processor, under 600 bucks'
0.9647  |  'gaming laptop with NVIDIA graphics under $1200'  vs  'NVIDIA gaming laptop under $1200 budget'


## 9z. Bonus — FP-Growth "commonly paired specifications"

Loads precomputed association rules (from `bonus_fpgrowth.ipynb`) and attaches a
"commonly paired with" insight to each recommendation's top result, based on its CPU tier.
Degrades gracefully (empty list, no error) if the rules file isn't present in this environment.

In [33]:
FPGROWTH_RULES_PATH = os.path.join(DATA_DIR, "fpgrowth_association_rules.csv")

fpgrowth_rules = None
if os.path.exists(FPGROWTH_RULES_PATH):
    fpgrowth_rules = pd.read_csv(FPGROWTH_RULES_PATH)
    fpgrowth_rules["antecedent"] = fpgrowth_rules["antecedent"].apply(eval)
    fpgrowth_rules["consequent"] = fpgrowth_rules["consequent"].apply(eval)
    print(f"Loaded {len(fpgrowth_rules)} FP-Growth association rules")
else:
    print("fpgrowth_association_rules.csv not found — commonly_paired_specs will be empty. "
          "Upload it from bonus_fpgrowth.ipynb's output to enable this feature.")


def bucket_cpu_simple(cpu: str):
    if not isinstance(cpu, str):
        return None
    cpu_l = cpu.lower()
    if "ryzen 7" in cpu_l or "ryzen 9" in cpu_l:
        return "CPU_Ryzen7/9"
    if "ryzen 5" in cpu_l:
        return "CPU_Ryzen5"
    if "ryzen 3" in cpu_l:
        return "CPU_Ryzen3"
    if "i9" in cpu_l or "i7" in cpu_l:
        return "CPU_i7/i9"
    if "i5" in cpu_l:
        return "CPU_i5"
    if "i3" in cpu_l:
        return "CPU_i3"
    return None


def get_paired_specs(cpu_value: str, top_n: int = 3):
    if fpgrowth_rules is None:
        return []
    bucket = bucket_cpu_simple(cpu_value)
    if bucket is None:
        return []
    matches = fpgrowth_rules[
        fpgrowth_rules["antecedent"].apply(lambda a: bucket in a and len(a) == 1)
    ].sort_values("lift", ascending=False).head(top_n)
    return [
        {"paired_with": ", ".join(row["consequent"]), "confidence": round(row["confidence"], 2), "lift": round(row["lift"], 2)}
        for _, row in matches.iterrows()
    ]

# quick check
print(get_paired_specs("AMD Ryzen 7 5700U"))

Loaded 554 FP-Growth association rules
[]


## 9z-2. `recommend_full()` — combines grounded generation + semantic cache + FP-Growth insight

This is the function the live API (section 6) now calls, replacing the bare `recommend_verified()`.
Adds a `cache_hit` field (missing from the original schema) and a `commonly_paired_specs` field
(the FP-Growth bonus, previously not wired into the pipeline output at all).

In [34]:
def recommend_full(query: str, index_type, preset, k: int = 5, use_cache: bool = True):
    if use_cache:
        query_embedding = embed_model.encode([query])[0].astype("float32")
        cached_response, match_info = cache.lookup(query_embedding)
        if cached_response is not None:
            result = dict(cached_response)
            result["cache_hit"] = True
            result["matched_query"] = match_info["matched_query"]
            result["similarity"] = round(match_info["similarity"], 4)
            return result

    result = recommend_verified(query, index_type, preset, k)
    result["cache_hit"] = False

    top_cpu = result["retrieved_context"][0]["cpu"] if result["retrieved_context"] else None
    result["commonly_paired_specs"] = get_paired_specs(top_cpu) if top_cpu else []

    if use_cache:
        cache.store(query, query_embedding, result)

    return result

# quick check
test_result = recommend_full("AMD Ryzen laptop under $600", "flat", "max", use_cache=False)
print("cache_hit:", test_result["cache_hit"])
print("commonly_paired_specs:", test_result["commonly_paired_specs"])

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: flat/max
cache_hit: False
commonly_paired_specs: []


All output files (`m4_retrieval_evaluation.csv`, `m4_qualitative_samples.json`,
`semantic_cache_benchmark.csv`, `semantic_cache_summary.csv`, `semantic_cache_threshold_sweep.csv`,
`semantic_cache_paraphrase_similarities.csv`) land in `/kaggle/working/` and are downloadable from
the notebook's Output tab once it finishes running — no `google.colab.files.download` needed on
Kaggle.


## 6. (Optional) Expose as a real FastAPI web service via ngrok

Kaggle notebooks *can* run a live HTTP endpoint the same way Colab does, as long as Internet is
enabled in Settings. Get a free ngrok authtoken at ngrok.com/signup. If you're presenting the M5
frontend separately and just need the pipeline logic (the cells above), you can skip this section
entirely and call `recommend_verified()` directly.


In [35]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "pyngrok", "nest_asyncio"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'pyngrok', 'nest_asyncio'], returncode=0)

In [36]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok
import threading

In [37]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("NGROK_TOK")

In [38]:
ngrok.set_auth_token(secret_value_0)

In [39]:
class IndexType(str, Enum):
    flat = "flat"
    ivfflat = "ivfflat"
    pq = "pq"
    ivfpq = "ivfpq"
    hnsw = "hnsw"

class Query(BaseModel):
    query: str
    k: int = 5
    index_type: IndexType = IndexType.ivfpq
    preset: IndexPreset = IndexPreset.MEDIUM
    use_cache: bool = True

In [40]:
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="RagBottom: Laptop RAG Recommender")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # tighten this later to your actual Flutter web origin
    allow_credentials=True,
    allow_methods=["*"],          # must include OPTIONS
    allow_headers=["*"],
)

In [41]:
@app.get("/")
def root():
    return {
        "status": "ok",
        "message": (
            "POST to /recommend with "
            "{'query': '...', 'index_type': 'ivfpq', "
            "'preset': 'high'}"
        )
    }

In [42]:
@app.post("/recommend")
def api_recommend(q: Query):
    return recommend_full(
        query=q.query,
        k=q.k,
        index_type=q.index_type.value,
        preset=q.preset.value,
        use_cache=q.use_cache,
    )

In [43]:
for route in app.routes:
    print(route.path, route.methods)

/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/ {'GET'}
/recommend {'POST'}


In [46]:
nest_asyncio.apply()
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print("Public URL:", PUBLIC_URL)
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True
).start()

Public URL: https://unmade-prognosis-savanna.ngrok-free.dev


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.180.245.250:0 - "OPTIONS /recommend HTTP/1.1" 200 OK


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Using cached index: flat/high
INFO:     103.180.245.250:0 - "POST /recommend HTTP/1.1" 200 OK


In [45]:
# import requests
# response = requests.post(
#     f"{PUBLIC_URL}/recommend",
#     json={
#         "query": "gaming laptop with NVIDIA graphics under $1200",
#         "k": 5,
#         "index_type": "ivfpq",
#         "preset": "high",
#         "use_cache": True,
#     }
# )
# print(response.status_code)
# print(response.text)